In [0]:
-- TODO: manual review required — choose a BTEQ replacement wrapper: Databricks notebook, stored procedure, or Workflows DAG
-- TODO: manual review required — BTEQ .IF ERRORCODE / .QUIT handling must be implemented by automatic procedure failure or Workflows branching

/**************************************************************************\
* DERIVE THE COUNTERPARTY KEY FOR EVERY TRANSACTION                        *
* ------------------------------------------------------------------------ *
* The counterparty account key is the IBAN where delivered, otherwise the  *
* UK sort code and account number. The name is normalised with the same    *
* rule as the sanctions list (kycref.60) so that both can be compared.     *
* Card, cash and fee transactions without a counterparty account are not   *
* assigned a counterparty.                                                 *
\**************************************************************************/

-- TODO: manual review required — validate fixed-width CHAR padding and bounded VARCHAR behavior against downstream consumers
-- Converted Teradata volatile-table semantics to a Databricks session-scoped temporary table; removed PRIMARY INDEX and ON COMMIT PRESERVE ROWS
CREATE TEMP TABLE VT_DLYTXN_CPTY_KEY
(
       TXN_REF            VARCHAR(35)  NOT NULL,
       SUPPLIER_CD        CHAR(4)      NOT NULL,
       BOOKING_DT         DATE,
       CPTY_ACCT_KEY      VARCHAR(34),
       CPTY_NORM_NM       VARCHAR(140),
       CPTY_BIC_CD        CHAR(11),
       CPTY_CTRY_CD       CHAR(2),
       CPTY_BANK_CTRY_CD  CHAR(2),
       CPTY_UK_ACCT_ID    CHAR(14)
);

-- Converted OREPLACE/OTRANSLATE to REPLACE/TRANSLATE, preserved Teradata trailing-space comparison semantics, and removed the unmapped source database qualifier
-- TODO: manual review required — verify that Databricks CHAR(14) casts preserve the required fixed-width padding at these expression boundaries
INSERT INTO VT_DLYTXN_CPTY_KEY
SELECT dtl.TXN_REF,
       dtl.SUPPLIER_CD,
       dtl.BOOKING_DT,
       COALESCE (NULLIF (TRIM (dtl.CPTY_IBAN), ''),
                 NULLIF (TRIM (dtl.CPTY_SORT_CD || dtl.CPTY_ACCT_NR), '')),
       COALESCE (NULLIF (TRIM (REPLACE (REPLACE (TRANSLATE (UPPER (TRIM (dtl.CPTY_NM)), '.,''-&/()', '        '), '  ', ' '), '  ', ' ')), ''),
                 'UNKNOWN'),
       CASE WHEN RTRIM (dtl.CPTY_BIC) = RTRIM (' ') THEN NULL ELSE dtl.CPTY_BIC END,
       CASE WHEN RTRIM (dtl.CPTY_CTRY) = RTRIM (' ') THEN NULL ELSE dtl.CPTY_CTRY END,
       CASE
          WHEN RTRIM (dtl.CPTY_BANK_CTRY) <> RTRIM (' ')      THEN dtl.CPTY_BANK_CTRY
          WHEN CHARACTER_LENGTH (TRIM (dtl.CPTY_BIC)) = 11    THEN SUBSTR (dtl.CPTY_BIC, 5, 2)
          WHEN dtl.CPTY_IBAN IS NOT NULL                      THEN SUBSTR (dtl.CPTY_IBAN, 1, 2)
          WHEN TRIM (dtl.CPTY_SORT_CD) <> ''                  THEN 'GB'
          ELSE NULL
       END,
       /* UK IBAN: GBkk BBBB SSSSSS AAAAAAAA -> sort code / account number */
       CASE
          WHEN RTRIM (SUBSTR (dtl.CPTY_IBAN, 1, 2)) = RTRIM ('GB')
           AND CHARACTER_LENGTH (dtl.CPTY_IBAN) = 22
          THEN CAST (SUBSTR (dtl.CPTY_IBAN, 9, 14) AS CHAR(14))
          WHEN TRIM (dtl.CPTY_SORT_CD) <> ''
           AND TRIM (dtl.CPTY_ACCT_NR) <> ''
          THEN CAST (dtl.CPTY_SORT_CD || dtl.CPTY_ACCT_NR AS CHAR(14))
          ELSE NULL
       END
FROM
       DLYTXN_LOAD   dtl
WHERE
       COALESCE (NULLIF (TRIM (dtl.CPTY_IBAN), ''),
                 NULLIF (TRIM (dtl.CPTY_SORT_CD || dtl.CPTY_ACCT_NR), '')) IS NOT NULL
;

-- Converted the deliberate pre-query column-statistics refresh to Databricks ANALYZE TABLE
ANALYZE TABLE VT_DLYTXN_CPTY_KEY COMPUTE STATISTICS FOR COLUMNS CPTY_ACCT_KEY, CPTY_NORM_NM;
-- Converted the deliberate pre-query column-statistics refresh to Databricks ANALYZE TABLE
ANALYZE TABLE VT_DLYTXN_CPTY_KEY COMPUTE STATISTICS FOR COLUMNS CPTY_UK_ACCT_ID;
-- Removed Teradata index statistics because Delta has no indexes and maintains table-level statistics automatically


/**************************************************************************\
* COLLECT COUNTERPARTIES NOT SEEN BEFORE                                   *
* ------------------------------------------------------------------------ *
* One row per account key and normalised name. Where the same counterparty *
* arrives with different BIC / country in one file, the latest booking     *
* wins.                                                                    *
\**************************************************************************/

-- Removed the unmapped Teradata database qualifier
DELETE FROM DLYTXN_NEW_CPTY
;

-- Removed unmapped Teradata database qualifiers and preserved trailing-space equivalence in character joins and window partitions
INSERT INTO DLYTXN_NEW_CPTY
(
       CPTY_ACCT_KEY,
       CPTY_NORM_NM,
       CPTY_BIC_CD,
       CPTY_CTRY_CD,
       FIRST_SEEN_DT
)
SELECT vk.CPTY_ACCT_KEY,
       vk.CPTY_NORM_NM,
       vk.CPTY_BIC_CD,
       COALESCE (vk.CPTY_CTRY_CD, vk.CPTY_BANK_CTRY_CD),
       MIN (vk.BOOKING_DT) OVER (PARTITION BY vk.CPTY_ACCT_KEY COLLATE UTF8_BINARY_RTRIM,
                                              vk.CPTY_NORM_NM COLLATE UTF8_BINARY_RTRIM)
FROM
       VT_DLYTXN_CPTY_KEY            vk

       LEFT OUTER JOIN
       CPTY                           cpt
          ON  RTRIM (cpt.CPTY_ACCT_KEY) = RTRIM (vk.CPTY_ACCT_KEY)
          AND RTRIM (cpt.CPTY_NORM_NM)  = RTRIM (vk.CPTY_NORM_NM)
WHERE
       cpt.CPTY_ID IS NULL

QUALIFY ROW_NUMBER () OVER (PARTITION BY vk.CPTY_ACCT_KEY COLLATE UTF8_BINARY_RTRIM,
                                         vk.CPTY_NORM_NM COLLATE UTF8_BINARY_RTRIM
                                ORDER BY vk.BOOKING_DT DESC,
                                         vk.TXN_REF DESC
                           ) = 1
;


/**************************************************************************\
* ASSIGN SURROGATE KEYS TO NEW COUNTERPARTIES                              *
* ------------------------------------------------------------------------ *
* New CPTY_ID = highest existing key + running count over the new rows.    *
\**************************************************************************/

-- Converted CSUM to an ANSI cumulative window, ZEROIFNULL to COALESCE, and removed unmapped Teradata database qualifiers
INSERT INTO CPTY
(
       CPTY_ID,
       CPTY_ACCT_KEY,
       CPTY_NORM_NM,
       CPTY_BIC_CD,
       CPTY_CTRY_CD,
       FIRST_SEEN_DT,
       LAST_SEEN_DT,
       TXN_CNT,
       CO_UPD_TMS
)
SELECT mx.MAX_CPTY_ID
         + SUM (1) OVER (
               ORDER BY ncp.CPTY_ACCT_KEY COLLATE UTF8_BINARY_RTRIM,
                        ncp.CPTY_NORM_NM COLLATE UTF8_BINARY_RTRIM
               ROWS UNBOUNDED PRECEDING
           ),
       ncp.CPTY_ACCT_KEY,
       ncp.CPTY_NORM_NM,
       ncp.CPTY_BIC_CD,
       ncp.CPTY_CTRY_CD,
       ncp.FIRST_SEEN_DT,
       ncp.FIRST_SEEN_DT,
       0,
       tms.LOAD_TMS
FROM
       DLYTXN_NEW_CPTY   ncp

       CROSS JOIN
       (SELECT COALESCE (MAX (CPTY_ID), 0) AS MAX_CPTY_ID
        FROM   CPTY
       )                   mx

       CROSS JOIN
       DLYTXN_LOAD_TMS     tms
;


/**************************************************************************\
* MAINTAIN LAST SEEN DATE AND TRANSACTION COUNT                            *
\**************************************************************************/

-- Converted Teradata UPDATE FROM to Databricks MERGE, ZEROIFNULL to COALESCE, and removed unmapped database qualifiers
-- Character grouping and matching use trailing-space-insensitive semantics
MERGE INTO CPTY AS cpt
USING (
       SELECT agg.CPTY_ACCT_KEY,
              agg.CPTY_NORM_NM,
              agg.LAST_SEEN_DT,
              agg.TXN_CNT,
              agg.CPTY_BIC_CD,
              tms.LOAD_TMS
       FROM   (SELECT CPTY_ACCT_KEY COLLATE UTF8_BINARY_RTRIM AS CPTY_ACCT_KEY,
                      CPTY_NORM_NM COLLATE UTF8_BINARY_RTRIM AS CPTY_NORM_NM,
                      MAX (BOOKING_DT)   AS LAST_SEEN_DT,
                      COUNT (*)          AS TXN_CNT,
                      MAX (CPTY_BIC_CD)  AS CPTY_BIC_CD
               FROM   VT_DLYTXN_CPTY_KEY
               GROUP  BY CPTY_ACCT_KEY COLLATE UTF8_BINARY_RTRIM,
                         CPTY_NORM_NM COLLATE UTF8_BINARY_RTRIM
              ) agg
              CROSS JOIN DLYTXN_LOAD_TMS tms
      ) AS src
ON     RTRIM (cpt.CPTY_ACCT_KEY) = RTRIM (src.CPTY_ACCT_KEY)
AND    RTRIM (cpt.CPTY_NORM_NM)  = RTRIM (src.CPTY_NORM_NM)
WHEN MATCHED THEN UPDATE SET
       LAST_SEEN_DT = CASE WHEN src.LAST_SEEN_DT > cpt.LAST_SEEN_DT
                           THEN src.LAST_SEEN_DT
                           ELSE cpt.LAST_SEEN_DT END,
       TXN_CNT      = COALESCE (cpt.TXN_CNT, 0) + src.TXN_CNT,
       CPTY_BIC_CD  = COALESCE (cpt.CPTY_BIC_CD, src.CPTY_BIC_CD),
       CO_UPD_TMS   = src.LOAD_TMS
;


/**************************************************************************\
* RESOLVE COUNTERPARTY PER TRANSACTION                                     *
* ------------------------------------------------------------------------ *
* Counterparties holding an account with the bank are flagged as internal  *
* and linked to the primary party of that account. This allows the         *
* transaction monitoring to follow funds between own customers (mule       *
* chains, layering through related accounts).                              *
\**************************************************************************/

-- Removed the unmapped Teradata database qualifier
DELETE FROM DLYTXN_CPTY
;

-- Removed unmapped Teradata database qualifiers and preserved trailing-space equivalence in all character joins
INSERT INTO DLYTXN_CPTY
(
       TXN_REF,
       SUPPLIER_CD,
       CPTY_ID,
       CPTY_ACCT_KEY,
       CPTY_NORM_NM,
       CPTY_CTRY_CD,
       CPTY_BANK_CTRY_CD,
       CPTY_INTERNAL_IND,
       CPTY_ACCT_ID,
       CPTY_PARTY_ID,
       NEW_CPTY_IND
)
SELECT vk.TXN_REF,
       vk.SUPPLIER_CD,
       cpt.CPTY_ID,
       vk.CPTY_ACCT_KEY,
       vk.CPTY_NORM_NM,
       COALESCE (vk.CPTY_CTRY_CD, vk.CPTY_BANK_CTRY_CD),
       vk.CPTY_BANK_CTRY_CD,
       CASE WHEN act.ACCT_ID IS NOT NULL THEN 'Y' ELSE 'N' END,
       act.ACCT_ID,
       act.PRIMARY_PARTY_ID,
       CASE WHEN ncp.CPTY_ACCT_KEY IS NOT NULL THEN 'Y' ELSE 'N' END
FROM
       VT_DLYTXN_CPTY_KEY   vk

       LEFT OUTER JOIN
       CPTY                 cpt
          ON  RTRIM (cpt.CPTY_ACCT_KEY) = RTRIM (vk.CPTY_ACCT_KEY)
          AND RTRIM (cpt.CPTY_NORM_NM)  = RTRIM (vk.CPTY_NORM_NM)

       LEFT OUTER JOIN
       DLYTXN_NEW_CPTY      ncp
          ON  RTRIM (ncp.CPTY_ACCT_KEY) = RTRIM (vk.CPTY_ACCT_KEY)
          AND RTRIM (ncp.CPTY_NORM_NM)  = RTRIM (vk.CPTY_NORM_NM)

       LEFT OUTER JOIN
       ACCT                  act
          ON  RTRIM (act.ACCT_ID) = RTRIM (vk.CPTY_UK_ACCT_ID)
;

-- Removed Teradata index statistics because Delta has no indexes and maintains table-level statistics automatically
-- Removed Teradata index statistics because Delta has no indexes and maintains table-level statistics automatically

-- Converted cleanup to DROP TEMPORARY TABLE for the migrated session-scoped table
DROP TEMPORARY TABLE VT_DLYTXN_CPTY_KEY
;

